In [14]:
import os
import json
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset
from torchvision import datasets, transforms, models
from PIL import Image

import numpy as np
import plotly.graph_objects as go

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit

In [15]:
user_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [16]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [17]:
DATA_DIR = Path('data')
TRAIN_DIR = DATA_DIR / 'Train'
TEST_DIR = DATA_DIR / 'Test'
VAL_DIR = DATA_DIR / 'Validation'

In [18]:
IMG_SIZE = 224
BATCH_SIZE = 32
LEARN_RATE = 3e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 20
PATIENCE = 5
VAL_SPLIT = 0.1

In [19]:
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(parents = True, exist_ok = True)

assert TRAIN_DIR.exists() and VAL_DIR.exists() and TEST_DIR.exists(), 'Train/Validation/Test folders not found.'
print('Using data at:', DATA_DIR.resolve())

Using data at: D:\Coding\Uni Marburg\gender-recogniser\data


In [20]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale = (0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness = 0.1, contrast = 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean = IMAGENET_MEAN, std = IMAGENET_STD)
])

eval_tf = transforms.Compose([
    transforms.Resize((256, 256)), 
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean = IMAGENET_MEAN, std = IMAGENET_STD)
])

ds_train = datasets.ImageFolder(TRAIN_DIR, transform = train_tf)
class_names = ds_train.classes
num_classes = len(class_names)

y_full = np.array([y for _, y in ds_train.samples])
strat_split = StratifiedShuffleSplit(n_splits = 1, test_size = VAL_SPLIT, random_state = SEED)
train_idx, val_idx = next(strat_split.split(np.zeros(len(y_full)), y_full))

ds_train = Subset(datasets.ImageFolder(TRAIN_DIR, transform = train_tf), train_idx)
ds_val = Subset(datasets.ImageFolder(TRAIN_DIR, transform = eval_tf), val_idx)
ds_test = datasets.ImageFolder(TEST_DIR, transform = eval_tf)


In [23]:
print('Classes:', class_names, ' (num_classes =', num_classes, ')')

print(f'Train: {len(ds_train)} | Val: {len(ds_val)} | Test: {len(ds_test)}')

Classes: ['Female', 'Male']  (num_classes = 2 )
Train: 10021 | Val: 1114 | Test: 1279


In [24]:
class_counts = np.bincount(y_full, minlength = num_classes)
class_weights = 1.0 / np.maximum(class_counts, 1.0)

In [25]:
train_targets = y_full[train_idx]
sample_weights = [class_weights[y] for y in train_targets]

sampler = WeightedRandomSampler(weights = [float(w) for w in sample_weights],
                                num_samples = len(sample_weights), replacement = True)

dl_train = DataLoader(ds_train, batch_size = BATCH_SIZE, sampler = sampler, num_workers = 2, pin_memory = True)
dl_test = DataLoader(ds_test, batch_size = BATCH_SIZE, shuffle = False, num_workers = 2, pin_memory = True)
dl_val = DataLoader(ds_val, batch_size = BATCH_SIZE, shuffle = False, num_workers = 2, pin_memory = True)

In [ ]:
try:
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights = weights)
except Exception:
    model = models.resnet18(pretrained = True)

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)k
model = model.to(user_device)

c_weight = torch.Tensor(class_weights / class_weights.sum() * num_classes, dtype = torch.float32).to(user_device)
criterion = nn.CrossEntropyLoss(weight = c_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr = LEARN_RATE, weight_decay = WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode = 'min', factor = 0.5, patience = 5)